# Drosophila Neural Simulation → Flybody Integration

This notebook:
1. Runs a spiking neural network simulation of the Drosophila connectome (138,639 LIF neurons, 15M synapses)
2. **Stimulates sensory neuron subsets** via Poisson input
3. Records spikes from **descending motor neurons** (P9, DNa01/02, MDN, Giant Fiber, MN9, aDN1)
4. Maps motor neuron activity to **flybody** actions using CPG-modulated behavioral primitives
5. Records a **video** of the fly body moving in response to the neural simulation

**Prerequisites:**
- SONATA circuit files in `./sonata_circuit/` (run `convert_to_sonata.py` first)
- `torch`, `h5py`, `dm_control`, `flybody` installed

In [ ]:
import json
import sys
from pathlib import Path
from time import perf_counter

import h5py
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn

# Ensure flybody is importable
FLYBODY_PATH = Path("../../../flybody")
if str(FLYBODY_PATH.resolve()) not in sys.path:
    sys.path.insert(0, str(FLYBODY_PATH.resolve()))

CIRCUIT_DIR = Path("./sonata_circuit")
OUTPUT_DIR = Path("../../../obi-output/drosophila_flybody")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

## 1. Experiment Configuration

Choose which sensory neurons to stimulate during the simulation.
Each scenario activates different descending neuron populations, producing different behaviors.

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Timing
# ──────────────────────────────────────────────────────────────────────
SPONTANEOUS_MS = 0.0      # No spontaneous period — stimulate immediately
STIMULATED_MS  = 2000.0   # 2 s with sensory stimulation
TOTAL_MS = SPONTANEOUS_MS + STIMULATED_MS

# ──────────────────────────────────────────────────────────────────────
# Descending motor neurons (from fly-brain/code/paper-phil-drosophila/example.ipynb)
# ──────────────────────────────────────────────────────────────────────
MOTOR_NEURONS = {
    "P9_left": 720575940627652358, "P9_right": 720575940635872101,
    "P9_oDN1_left": 720575940626730883, "P9_oDN1_right": 720575940620300308,
    "DNa01_left": 720575940644438551, "DNa01_right": 720575940627787609,
    "DNa02_left": 720575940604737708, "DNa02_right": 720575940629327659,
    "MDN_1": 720575940616026939, "MDN_2": 720575940631082808,
    "MDN_3": 720575940640331472, "MDN_4": 720575940610236514,
    "Giant_Fiber_1": 720575940622838154, "Giant_Fiber_2": 720575940632499757,
    "MN9_left": 720575940660219265, "MN9_right": 720575940618238523,
    "aDN1_left": 720575940624319124, "aDN1_right": 720575940616185531,
}

# ──────────────────────────────────────────────────────────────────────
# Sensory neuron groups for stimulation
# ──────────────────────────────────────────────────────────────────────
SENSORY_GROUPS = {
    "P9s": {
        "ids": [720575940627652358, 720575940635872101],
        "rate_hz": 100.0,
        "description": "P9 forward-walking neurons (100 Hz) → forward locomotion",
    },
    "sugar_GRNs": {
        "ids": [
            720575940616885538, 720575940630233916, 720575940639332736,
            720575940632889389, 720575940617000768, 720575940632425919,
            720575940637568838, 720575940629176663, 720575940621502051,
            720575940638202345, 720575940612670570, 720575940611875570,
            720575940621754367, 720575940633143833, 720575940613601698,
            720575940630797113, 720575940639198653, 720575940639259967,
            720575940624963786, 720575940640649691, 720575940610788069,
            720575940623172843, 720575940628853239,
        ],
        "rate_hz": 200.0,
        "description": "Sugar gustatory receptor neurons (200 Hz) → feeding",
    },
    "LC4s": {
        "ids": [
            720575940605598892, 720575940611134833, 720575940612580977,
            720575940613256863, 720575940613260959, 720575940614914107,
            720575940615462587, 720575940617176321, 720575940617266722,
            720575940618807105, 720575940620795728, 720575940622108001,
            720575940624017251, 720575940625038090, 720575940625934973,
            720575940625991043, 720575940626605200, 720575940626626895,
            720575940628454522, 720575940628462340, 720575940630851036,
            720575940638496720, 720575940603637438, 720575940610522009,
            720575940612093351, 720575940612323025, 720575940612380723,
            720575940612498129, 720575940612518055, 720575940612968421,
            720575940613609484, 720575940613638041, 720575940614572742,
            720575940614582946, 720575940615053580, 720575940615127227,
            720575940615232217, 720575940615575007, 720575940616066705,
            720575940616713355, 720575940617026260, 720575940617348379,
            720575940618002644, 720575940618234704, 720575940618234715,
            720575940618266459, 720575940618267227, 720575940618275520,
            720575940618312606, 720575940618676440, 720575940618709158,
            720575940618723749, 720575940619397542, 720575940620314221,
            720575940620314612, 720575940620731380, 720575940620903551,
            720575940621145821, 720575940621522458, 720575940621753579,
            720575940622330582, 720575940622531767, 720575940622939836,
            720575940624111763, 720575940624790781, 720575940624856762,
            720575940625841351, 720575940625845447, 720575940625906702,
            720575940625932421, 720575940626553596, 720575940626916936,
            720575940627519107, 720575940628064260, 720575940628081541,
            720575940628419527, 720575940628518400, 720575940628599895,
            720575940628606713, 720575940628699560, 720575940628891863,
            720575940629753807, 720575940629964591, 720575940630154660,
            720575940630484495, 720575940630998339, 720575940631032657,
            720575940631338271, 720575940632475449, 720575940632715234,
            720575940632769180, 720575940633013355, 720575940633218863,
            720575940633580384, 720575940634517856, 720575940635835967,
            720575940636957006, 720575940638456227, 720575940639817947,
            720575940640612480, 720575940641213824, 720575940645821316,
            720575940649229433, 720575940652611745,
        ],
        "rate_hz": 200.0,
        "description": "LC4 looming-sensitive visual neurons (200 Hz) → escape",
    },
}

# ──────────────────────────────────────────────────────────────────────
# ▶ PER-FLY BRAIN STIMULATION
# Each fly has its own brain (same 138K-neuron circuit, different input)
# ──────────────────────────────────────────────────────────────────────
FLY_BRAINS = {
    "Fly 1": {
        "experiments": ["P9s"],
        "description": "P9 stimulation → forward locomotion",
    },
    "Fly 2": {
        "experiments": ["P9s", "LC4s"],
        "description": "P9 + LC4 stimulation → walking interrupted by escape",
    },
}

for fly, cfg in FLY_BRAINS.items():
    n_stim = sum(len(SENSORY_GROUPS[e]["ids"]) for e in cfg["experiments"])
    print(f"{fly}: {cfg['description']} ({n_stim} sensory neurons)")
print(f"\nSimulation: {TOTAL_MS:.0f} ms per brain × 2 brains")

## 2. Load SONATA Circuit

In [ ]:
def load_sonata_circuit(circuit_dir: Path) -> dict:
    """Load SONATA circuit data from HDF5 files."""
    net = circuit_dir / "network"
    data = {}

    with open(circuit_dir / "circuit_config.json") as f:
        data["circuit_config"] = json.load(f)
    with open(circuit_dir / "simulation_config.json") as f:
        data["sim_config"] = json.load(f)

    # Internal nodes
    with h5py.File(net / "internal_nodes.h5", "r") as f:
        pop = f["nodes/internal"]
        data["n_neurons"] = len(pop["node_id"])
        data["flywire_ids"] = pop["0/flywire_id"][:]

    # Internal edges (recurrent synapses)
    with h5py.File(net / "internal_internal_edges.h5", "r") as f:
        pop = f["edges/internal_to_internal"]
        data["pre_idx"] = pop["source_node_id"][:].astype(np.int64)
        data["post_idx"] = pop["target_node_id"][:].astype(np.int64)
        data["weights"] = pop["0/syn_weight"][:]
        data["delays"] = pop["0/delay"][:]

    # Neuron dynamics parameters
    params_path = circuit_dir / "components" / "cell_models" / "flywire_lif.json"
    with open(params_path) as f:
        data["neuron_params"] = json.load(f)

    data["dt"] = data["sim_config"]["run"]["dt"]

    # Build FlyWire ID ↔ node_id maps
    data["fw_to_node"] = {int(fw): i for i, fw in enumerate(data["flywire_ids"])}
    data["node_to_fw"] = {i: int(fw) for i, fw in enumerate(data["flywire_ids"])}

    return data


t0 = perf_counter()
circuit = load_sonata_circuit(CIRCUIT_DIR)
n = circuit["n_neurons"]
dt = circuit["dt"]
print(f"Loaded in {perf_counter() - t0:.1f}s")
print(f"  Neurons: {n:,}")
print(f"  Synapses: {len(circuit['weights']):,}")
print(f"  dt: {dt} ms")

## 3. Resolve Neuron IDs to Node Indices

Map FlyWire IDs from the experiment config to SONATA node indices.

In [ ]:
fw_to_node = circuit["fw_to_node"]

# Resolve motor neuron FlyWire IDs → node indices
motor_node_ids = {}
for name, fw_id in MOTOR_NEURONS.items():
    if fw_id in fw_to_node:
        motor_node_ids[name] = fw_to_node[fw_id]
    else:
        print(f"  WARNING: {name} (fw={fw_id}) not found in circuit")

print(f"Motor neurons resolved: {len(motor_node_ids)}/{len(MOTOR_NEURONS)}")

# Build per-fly stimulation rate tensors
fly_stim_tensors = {}
for fly_name, cfg in FLY_BRAINS.items():
    rates = torch.zeros(n, device=DEVICE)
    total = 0
    for exp_name in cfg["experiments"]:
        group = SENSORY_GROUPS[exp_name]
        for fw_id in group["ids"]:
            if fw_id in fw_to_node:
                rates[fw_to_node[fw_id]] = group["rate_hz"]
                total += 1
    fly_stim_tensors[fly_name] = rates
    print(f"  {fly_name}: {total} stimulated neurons")

## 4. Build PyTorch Spiking Network Model

LIF neurons with alpha-function synapses, matching the SONATA/FlyWire circuit dynamics.

In [ ]:
class PoissonSpikeGenerator(nn.Module):
    def __init__(self, dt, scale, device="cpu"):
        super().__init__()
        self.prob_scale = dt / 1000.0
        self.scale = scale
        self.device = device

    def forward(self, rates):
        return torch.bernoulli(rates * self.prob_scale) * self.scale


class AlphaSynapse(nn.Module):
    def __init__(self, size, dt, tau_syn, delay_ms, device="cpu"):
        super().__init__()
        self.time_factor = dt / tau_syn
        self.steps_delay = int(delay_ms / dt)
        self.size = size
        self.device = device

    def state_init(self):
        conductance = torch.zeros(self.size, device=self.device)
        delay_buffer = torch.zeros(
            self.steps_delay + 1, self.size, device=self.device
        )
        return conductance, delay_buffer

    def forward(self, input_, conductance, delay_buffer, refrac):
        conductance_new = (
            conductance * (1 - self.time_factor) + delay_buffer[0, :] * refrac
        )
        delay_buffer = torch.roll(delay_buffer, shifts=-1, dims=0)
        delay_buffer[-1, :] = input_
        return conductance_new, delay_buffer


class LIFNeuron(nn.Module):
    def __init__(self, size, dt, params, device="cpu"):
        super().__init__()
        self.size = size
        self.time_factor = dt / params["tauMem"]
        self.v_reset = params["vReset"]
        self.v_rest = params["vRest"]
        self.v_threshold = params["vThreshold"]
        self.v_0 = params["v0"]
        self.device = device

    def state_init(self):
        v = torch.zeros(self.size, device=self.device) + self.v_0
        spikes = torch.zeros(self.size, device=self.device)
        return spikes, v

    def forward(self, input_current, v):
        v = v + self.time_factor * (input_current - (v - self.v_rest))
        spike = (v > self.v_threshold).float()
        reset = ((v - self.v_reset) * spike).detach()
        v = v - reset
        return spike, v


class AlphaLIF(nn.Module):
    def __init__(self, size, dt, params, device="cpu"):
        super().__init__()
        self.synapse = AlphaSynapse(
            size, dt, params["tauSyn"], params["tDelay"], device=device
        )
        self.neuron = LIFNeuron(size, dt, params, device=device)
        self.steps_refrac = int(params["tRefrac"] / dt)

    def state_init(self):
        conductance, delay_buffer = self.synapse.state_init()
        spikes, v = self.neuron.state_init()
        refrac = self.steps_refrac + torch.zeros_like(v)
        return conductance, delay_buffer, spikes, v, refrac

    def forward(self, input_, conductance, delay_buffer, spikes, v, refrac):
        refrac = refrac * (1 - spikes) + 1
        conductance_new, delay_buffer = self.synapse(
            input_, conductance, delay_buffer,
            (refrac > self.steps_refrac).float(),
        )
        spikes, v_new = self.neuron(conductance, v)
        conductance_reset = (conductance_new * spikes).detach()
        conductance_new = conductance_new - conductance_reset
        return conductance_new, delay_buffer, spikes, v_new, refrac


class SonataModel(nn.Module):
    def __init__(self, size, dt, params, weights_sparse, device="cpu"):
        super().__init__()
        self.neurons = AlphaLIF(size, dt, params, device=device)
        self.weights = weights_sparse
        self.poisson = PoissonSpikeGenerator(
            dt, params["scalePoisson"], device=device
        )
        self.w_scale = params["wScale"]

    def state_init(self):
        return self.neurons.state_init()

    def forward(self, rates, conductance, delay_buffer, spikes, v, refrac):
        spikes_input = self.poisson(rates)
        weighted_spikes = torch.mv(self.weights, spikes)
        conductance, delay_buffer, spikes, v, refrac = self.neurons(
            self.w_scale * (spikes_input + weighted_spikes),
            conductance, delay_buffer, spikes, v, refrac,
        )
        return conductance, delay_buffer, spikes, v, refrac

In [ ]:
# Map SONATA neuron params → PyTorch model params
np_ = circuit["neuron_params"]
stim_weight = 68.75  # wScale * scalePoisson = 0.275 * 250

params = {
    "tauSyn": np_["tau_syn_ex"],
    "tDelay": float(circuit["delays"][0]),
    "v0": np_["E_L"],
    "vReset": np_["V_reset"],
    "vRest": np_["E_L"],
    "vThreshold": np_["V_th"],
    "tauMem": np_["tau_m"],
    "tRefrac": np_["t_ref"],
    "scalePoisson": stim_weight,
    "wScale": 1.0,
}

num_steps = int(TOTAL_MS / dt)
stim_onset_step = int(SPONTANEOUS_MS / dt)

# Build sparse weight matrix
print("Building sparse weight matrix...")
t0 = perf_counter()
weight_coo = torch.sparse_coo_tensor(
    torch.tensor(
        np.array([circuit["post_idx"], circuit["pre_idx"]]),
        dtype=torch.long,
    ),
    torch.tensor(circuit["weights"], dtype=torch.float32),
    size=(n, n),
).coalesce()
weights_sparse = weight_coo.to_sparse_csr().to(DEVICE)
print(f"  Built in {perf_counter() - t0:.2f}s  ({n:,}×{n:,}, {len(circuit['weights']):,} nnz)")

model = SonataModel(n, dt, params, weights_sparse, device=DEVICE)
print(f"\nSimulation: {TOTAL_MS:.0f} ms ({num_steps:,} steps)")
print(f"  Spontaneous:  0 – {SPONTANEOUS_MS:.0f} ms (step 0 – {stim_onset_step:,})")
print(f"  Stimulated:   {SPONTANEOUS_MS:.0f} – {TOTAL_MS:.0f} ms (step {stim_onset_step:,} – {num_steps:,})")

## 5. Run Two-Brain Neural Simulation (Co-simulated)

Both fly brains (same 138,639-neuron circuit, different sensory input) are
stepped together in lock-step. Each brain maintains its own membrane voltages,
synaptic conductances, and spike state — only the Poisson input differs.

In [ ]:
# ── Co-simulate two fly brains in lock-step ──
node_to_motor = {nid: name for name, nid in motor_node_ids.items()}
num_steps = int(TOTAL_MS / dt)
stim_onset_step = int(SPONTANEOUS_MS / dt)
rates_off = torch.zeros(n, device=DEVICE)

fly_names = list(FLY_BRAINS.keys())

# Initialize independent state for each brain
states = {}
for fly_name in fly_names:
    states[fly_name] = list(model.state_init())  # [conductance, delay_buffer, spikes, v, refrac]

# Per-brain recording
fly_results = {}
for fly_name in fly_names:
    fly_results[fly_name] = {
        "spike_times": [],
        "spike_ids": [],
        "motor_spike_trains": {name: [] for name in motor_node_ids},
    }

print(f"Co-simulating {len(fly_names)} brains × {TOTAL_MS:.0f} ms ({num_steps:,} steps)...")
for fly_name, cfg in FLY_BRAINS.items():
    print(f"  {fly_name}: {cfg['description']}")

t_sim = perf_counter()

with torch.no_grad():
    for t_step in range(num_steps):
        # Step each brain with its own input and state
        for fly_name in fly_names:
            stim = fly_stim_tensors[fly_name]
            rates = rates_off if t_step < stim_onset_step else stim
            c, db, sp, v, rf = states[fly_name]

            c, db, sp, v, rf = model(rates, c, db, sp, v, rf)
            states[fly_name] = [c, db, sp, v, rf]

            # Record spikes
            spike_mask = sp > 0
            if spike_mask.any():
                n_idx = spike_mask.nonzero(as_tuple=True)[0].cpu().numpy()
                t_ms = t_step * dt
                res = fly_results[fly_name]
                res["spike_times"].extend([t_ms] * len(n_idx))
                res["spike_ids"].extend(n_idx.tolist())
                for nid in n_idx:
                    motor_name = node_to_motor.get(int(nid))
                    if motor_name is not None:
                        res["motor_spike_trains"][motor_name].append(t_step)

        if num_steps >= 10000 and (t_step + 1) % (num_steps // 5) == 0:
            elapsed = perf_counter() - t_sim
            pct = (t_step + 1) / num_steps * 100
            print(f"  {pct:3.0f}% - {elapsed:.1f}s")

sim_time = perf_counter() - t_sim
print(f"\nDone in {sim_time:.1f}s")

# Finalize arrays and report
for fly_name in fly_names:
    res = fly_results[fly_name]
    res["spike_times"] = np.array(res["spike_times"])
    res["spike_ids"] = np.array(res["spike_ids"], dtype=np.int64)
    res["sim_time"] = sim_time

    n_total = len(res["spike_times"])
    n_active = len(set(res["spike_ids"])) if n_total > 0 else 0
    active_motors = {k: len(v) for k, v in res["motor_spike_trains"].items() if v}
    print(f"  {fly_name}: {n_total:,} spikes, {n_active:,} active neurons")
    if active_motors:
        print(f"    Motor: {active_motors}")
    else:
        print(f"    No motor spikes (synthetic activations will be used)")

## 5b. Raster Plots — Two Brains

Each fly brain receives different sensory input, producing distinct network and efferent activity.

- **Fly 1 efferents** (blue): P9, P9\_oDN1, DNa01, DNa02 — locomotion
- **Fly 2 efferents** (red): Giant Fiber, MDN, MN9, aDN1 — escape / grooming

In [ ]:
# ── Assign motor neurons to flies ──
FLY1_NEURONS = ["P9_left", "P9_right", "P9_oDN1_left", "P9_oDN1_right",
                "DNa01_left", "DNa01_right", "DNa02_left", "DNa02_right"]
FLY2_NEURONS = ["Giant_Fiber_1", "Giant_Fiber_2", "MDN_1", "MDN_2", "MDN_3", "MDN_4",
                "MN9_left", "MN9_right", "aDN1_left", "aDN1_right"]

fly1_node_ids = {motor_node_ids[n] for n in FLY1_NEURONS if n in motor_node_ids}
fly2_node_ids = {motor_node_ids[n] for n in FLY2_NEURONS if n in motor_node_ids}
all_motor_node_ids = fly1_node_ids | fly2_node_ids

# ── Build efferent spike arrays per brain ──
def get_efferent_spikes(motor_trains, neuron_list):
    times, ypos = [], []
    for i, name in enumerate(neuron_list):
        for t_step in motor_trains.get(name, []):
            times.append(t_step * dt)
            ypos.append(i)
    return times, ypos

# ── Figure: 2×2 grid — network raster + efferent raster per brain ──
fig, axes = plt.subplots(2, 2, figsize=(16, 10),
                         gridspec_kw={"height_ratios": [2.5, 1.2]},
                         sharex=True)
fly_colors = {"Fly 1": "royalblue", "Fly 2": "orangered"}

for col, (fly_name, res) in enumerate(fly_results.items()):
    color = fly_colors[fly_name]
    st, si = res["spike_times"], res["spike_ids"]
    mt = res["motor_spike_trains"]

    # ── Top: network raster ──
    ax = axes[0, col]
    max_pts = 300_000
    if len(st) > max_pts:
        idx = np.random.default_rng(col).choice(len(st), max_pts, replace=False)
        rt, ri = st[idx], si[idx]
    else:
        rt, ri = st, si

    is_eff1 = np.isin(ri, list(fly1_node_ids))
    is_eff2 = np.isin(ri, list(fly2_node_ids))
    is_eff = is_eff1 | is_eff2
    ax.scatter(rt[~is_eff], ri[~is_eff], s=0.05, alpha=0.2, c="0.4", rasterized=True)
    ax.scatter(rt[is_eff1], ri[is_eff1], s=8, alpha=0.8, c="royalblue", zorder=3)
    ax.scatter(rt[is_eff2], ri[is_eff2], s=8, alpha=0.8, c="orangered", zorder=3)
    ax.set_ylabel("Neuron index" if col == 0 else "")
    ax.set_ylim(-0.5, n)
    n_spk = len(st)
    n_act = len(set(si)) if len(si) > 0 else 0
    ax.set_title(f"{fly_name} Brain — {n_spk:,} spikes, {n_act:,} neurons",
                 fontsize=11, color=color)

    # ── Bottom: efferent raster for this fly's motor neurons ──
    ax_eff = axes[1, col]
    eff_neurons = FLY1_NEURONS if col == 0 else FLY2_NEURONS
    eff_t, eff_y = get_efferent_spikes(mt, eff_neurons)
    if eff_t:
        ax_eff.scatter(eff_t, eff_y, s=20, c=color, alpha=0.9, marker="|", linewidths=1.5)
    ax_eff.set_yticks(range(len(eff_neurons)))
    ax_eff.set_yticklabels(eff_neurons, fontsize=7)
    ax_eff.set_ylim(-0.5, len(eff_neurons) - 0.5)
    ax_eff.invert_yaxis()
    label = "locomotion" if col == 0 else "escape / groom"
    ax_eff.set_title(f"{fly_name} Efferents → {label}", fontsize=10, color=color)
    ax_eff.set_xlabel("Time (ms)")

for ax_row in axes:
    for ax in ax_row:
        ax.set_xlim(0, TOTAL_MS)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "raster_two_brains.png", dpi=150, bbox_inches="tight")
plt.show()

# Summary
for fly_name, res in fly_results.items():
    mt = res["motor_spike_trains"]
    active = {k: len(v) for k, v in mt.items() if v}
    print(f"{fly_name}: {len(res['spike_times']):,} total spikes, efferents: {active or 'none'}")

## 6. Compute Motor Neuron Activation Traces

Convert discrete spikes into smooth activation signals by convolving with an exponential kernel.  
Then group neurons by behavioral function and compute aggregate activations.

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Exponential smoothing of spike trains → continuous activation
# ──────────────────────────────────────────────────────────────────────
TAU_SMOOTH_MS = 100.0

def spike_steps_to_activation(spike_steps, num_steps, dt, tau_ms):
    """Convert spike timestep indices to a smooth activation trace."""
    decay = np.exp(-dt / tau_ms)
    trace = np.zeros(num_steps)
    activation = 0.0
    spike_set = set(spike_steps)
    for t in range(num_steps):
        activation *= decay
        if t in spike_set:
            activation += 1.0 / (tau_ms / 1000.0)
        trace[t] = activation
    return trace

BEHAVIOR_GROUPS = {
    "forward":  ["P9_left", "P9_right", "P9_oDN1_left", "P9_oDN1_right"],
    "turn_left":  ["DNa01_left", "DNa02_left"],
    "turn_right": ["DNa01_right", "DNa02_right"],
    "backward": ["MDN_1", "MDN_2", "MDN_3", "MDN_4"],
    "escape":   ["Giant_Fiber_1", "Giant_Fiber_2"],
    "feeding":  ["MN9_left", "MN9_right"],
    "grooming": ["aDN1_left", "aDN1_right"],
}

# Compute behavior activations per fly brain
fly_behavior_activations = {}
for fly_name, res in fly_results.items():
    mt = res["motor_spike_trains"]
    beh_act = {}
    for beh, neuron_names in BEHAVIOR_GROUPS.items():
        traces = [spike_steps_to_activation(mt[n], num_steps, dt, TAU_SMOOTH_MS)
                  for n in neuron_names if n in mt]
        beh_act[beh] = np.mean(traces, axis=0) if traces else np.zeros(num_steps)
        peak = beh_act[beh].max()
        if peak > 0:
            beh_act[beh] /= peak
    fly_behavior_activations[fly_name] = beh_act

for fly_name, beh_act in fly_behavior_activations.items():
    active = {b: f"{t.max():.2f}" for b, t in beh_act.items() if t.max() > 0.01}
    print(f"{fly_name} activations: {active or 'none (will use synthetic)'}")

In [ ]:
CONTROL_DT_MS = 2.0  # Flybody control timestep
NEURAL_STEPS_PER_CONTROL = int(CONTROL_DT_MS / dt)
N_CONTROL_STEPS = int(TOTAL_MS / CONTROL_DT_MS)

def downsample_activation(trace, neural_steps_per_control):
    n_control = len(trace) // neural_steps_per_control
    reshaped = trace[:n_control * neural_steps_per_control].reshape(n_control, neural_steps_per_control)
    return reshaped.mean(axis=1)

stim_onset_ctrl = int(SPONTANEOUS_MS / CONTROL_DT_MS)
stim_len = N_CONTROL_STEPS - stim_onset_ctrl

# Downsample per-fly activations to control timestep
fly1_beh = fly_behavior_activations["Fly 1"]
fly2_beh = fly_behavior_activations["Fly 2"]

fly1_act = {beh: downsample_activation(tr, NEURAL_STEPS_PER_CONTROL) for beh, tr in fly1_beh.items()}
fly2_act = {beh: downsample_activation(tr, NEURAL_STEPS_PER_CONTROL) for beh, tr in fly2_beh.items()}

# Check for sufficient activity, use synthetic fallback per fly
fly1_any = any(tr.max() > 0.01 for tr in fly1_act.values())
fly2_any = any(tr.max() > 0.01 for tr in fly2_act.values())

if not fly1_any:
    print("Fly 1: no motor spikes — using synthetic forward walking")
    fly1_act = {b: np.zeros(N_CONTROL_STEPS) for b in BEHAVIOR_GROUPS}
    fly1_act["forward"][stim_onset_ctrl:] = np.linspace(0.2, 1.0, stim_len)

if not fly2_any:
    print("Fly 2: no motor spikes — using synthetic escape + grooming")
    fly2_act = {b: np.zeros(N_CONTROL_STEPS) for b in BEHAVIOR_GROUPS}
    bl = int(300 / CONTROL_DT_MS)
    burst = np.exp(-np.arange(bl) * CONTROL_DT_MS / 80.)
    end = min(stim_onset_ctrl + bl, N_CONTROL_STEPS)
    fly2_act["escape"][stim_onset_ctrl:end] = burst[:end - stim_onset_ctrl]
    gs = stim_onset_ctrl + int(400 / CONTROL_DT_MS)
    if gs < N_CONTROL_STEPS:
        fly2_act["grooming"][gs:] = np.linspace(0.3, 1.0, N_CONTROL_STEPS - gs)

print(f"\nControl steps: {N_CONTROL_STEPS}")
for name, acts in [("Fly 1", fly1_act), ("Fly 2", fly2_act)]:
    active = {b: f"{t.max():.2f}" for b, t in acts.items() if t.max() > 0.01}
    print(f"  {name}: {active}")

In [ ]:
# Plot per-fly activation profiles over time
t_ms_ctrl = np.arange(N_CONTROL_STEPS) * CONTROL_DT_MS

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

# ── Fly 1 activations ──
ax = axes[0]
for beh, color in [("forward", "royalblue"), ("turn_left", "steelblue"), ("turn_right", "cornflowerblue")]:
    tr = fly1_act[beh]
    if tr.max() > 0.01:
        ax.fill_between(t_ms_ctrl, tr, alpha=0.5, color=color, label=beh)
ax.set_ylabel("Activation", fontsize=9)
ax.set_ylim(-0.05, 1.15)
ax.set_title("Fly 1 — Locomotion Activations", fontsize=10, color="royalblue")
ax.legend(loc="upper left", fontsize=8)

# ── Fly 2 activations ──
ax = axes[1]
for beh, color in [("escape", "orangered"), ("grooming", "goldenrod"),
                    ("backward", "salmon"), ("feeding", "sandybrown")]:
    tr = fly2_act[beh]
    if tr.max() > 0.01:
        ax.fill_between(t_ms_ctrl, tr, alpha=0.5, color=color, label=beh)
ax.set_ylabel("Activation", fontsize=9)
ax.set_ylim(-0.05, 1.15)
ax.set_title("Fly 2 — Defensive / Maintenance Activations", fontsize=10, color="orangered")
ax.legend(loc="upper left", fontsize=8)
ax.set_xlabel("Time (ms)")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "per_fly_activations.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Define Behavioral Action Primitives for Flybody

Each descending neuron group drives a stereotyped motor program. We define these as
CPG-like action patterns that modulate the 59-dimensional flybody action space.

**Action space layout** (walking config, wings disabled):
- `[0:6]` — Adhesion (6 claws)
- `[6:9]` — Head (abduct, twist, main)
- `[9:11]` — Abdomen (abduct, main)
- `[11:19]` — T1 left leg (coxa_abd, coxa_tw, coxa, fem_tw, fem, tib, tar, tar2)
- `[19:27]` — T1 right leg
- `[27:35]` — T2 left leg
- `[35:43]` — T2 right leg
- `[43:51]` — T3 left leg
- `[51:59]` — T3 right leg

In [ ]:
# Leg slice indices (each 8 DOFs: coxa_abd, coxa_tw, coxa, fem_tw, fem, tib, tar, tar2)
LEG_SLICES = {
    "T1_left":  slice(11, 19),
    "T1_right": slice(19, 27),
    "T2_left":  slice(27, 35),
    "T2_right": slice(35, 43),
    "T3_left":  slice(43, 51),
    "T3_right": slice(51, 59),
}
J_COXA_ABD, J_COXA_TW, J_COXA, J_FEM_TW, J_FEM, J_TIB, J_TAR, J_TAR2 = range(8)

TRIPOD_1 = ["T1_left", "T2_right", "T3_left"]
TRIPOD_2 = ["T1_right", "T2_left", "T3_right"]

N_ACTIONS = 59
WALK_FREQ_HZ = 10.0  # Brisk Drosophila gait


def tripod_walk_action(t_s, amplitude=1.0, freq=WALK_FREQ_HZ, direction=1.0):
    """Tripod gait. Uses large joint ranges for visible motion."""
    action = np.zeros(N_ACTIONS)
    omega = 2.0 * np.pi * freq
    action[:6] = 0.6  # adhesion

    for leg_group, phase_offset in [(TRIPOD_1, 0.0), (TRIPOD_2, np.pi)]:
        phase = omega * t_s + phase_offset
        for leg_name in leg_group:
            sl = LEG_SLICES[leg_name]
            a = amplitude
            # Coxa: big forward/back swing (range ~[-0.2, 1.7])
            action[sl.start + J_COXA] = 0.4 + direction * a * 0.8 * np.sin(phase)
            # Femur: high lift during swing (range ~[-0.15, 2.0])
            action[sl.start + J_FEM] = a * 0.9 * max(0, -np.cos(phase))
            # Tibia: extend during stance
            action[sl.start + J_TIB] = a * 0.6 * np.sin(phase + 0.3)
            # Tarsus: ground push
            action[sl.start + J_TAR] = a * 0.4 * np.sin(phase)
    return action


def turning_action(t_s, amplitude=1.0, turn_sign=1.0, freq=WALK_FREQ_HZ):
    """Turn by asymmetric stride + coxa abduction bias."""
    action = np.zeros(N_ACTIONS)
    omega = 2.0 * np.pi * freq
    action[:6] = 0.6

    for leg_group, phase_offset in [(TRIPOD_1, 0.0), (TRIPOD_2, np.pi)]:
        phase = omega * t_s + phase_offset
        for leg_name in leg_group:
            sl = LEG_SLICES[leg_name]
            is_left = "left" in leg_name
            # Slow the inner side, speed up outer
            side_scale = 0.2 if (is_left and turn_sign > 0) or (not is_left and turn_sign < 0) else 1.0
            a = amplitude * side_scale
            action[sl.start + J_COXA] = 0.4 + a * 0.8 * np.sin(phase)
            action[sl.start + J_FEM] = a * 0.8 * max(0, -np.cos(phase))
            action[sl.start + J_TIB] = a * 0.5 * np.sin(phase + 0.3)
            # Strong abduction bias
            action[sl.start + J_COXA_ABD] = turn_sign * amplitude * 0.3

    return action


def escape_action(t_s, amplitude=1.0):
    """Giant-fiber escape: explosive leg extension (jump-like)."""
    action = np.zeros(N_ACTIONS)
    action[:6] = 0.0  # release adhesion for takeoff

    for leg_name, sl in LEG_SLICES.items():
        a = amplitude
        action[sl.start + J_FEM] = a * 2.0     # max femur extension
        action[sl.start + J_TIB] = a * 1.3     # max tibia extension
        action[sl.start + J_COXA] = a * 1.5    # push back hard
        action[sl.start + J_TAR] = a * 1.0     # tarsus push
        action[sl.start + J_COXA_ABD] = a * 0.5  # splay legs out
    return action


def grooming_action(t_s, amplitude=1.0, freq=4.0):
    """Front legs reach up and sweep across head in large rhythmic arcs."""
    action = np.zeros(N_ACTIONS)
    action[:6] = 0.6
    phase = 2.0 * np.pi * freq * t_s
    a = amplitude

    # T1 legs: reach forward-up toward head with big sweeping motion
    for leg_name in ["T1_left", "T1_right"]:
        sl = LEG_SLICES[leg_name]
        sign = 1.0 if "left" in leg_name else -1.0
        action[sl.start + J_COXA] = 1.5 * a           # swing far forward
        action[sl.start + J_FEM] = 1.8 * a             # lift high
        action[sl.start + J_TIB] = a * 1.0 * np.sin(phase)  # big rhythmic sweep
        action[sl.start + J_TAR] = a * 0.6 * np.cos(phase)  # tarsus sweep
        action[sl.start + J_COXA_ABD] = sign * a * 0.4 * np.sin(phase + np.pi/3)  # lateral sweep
        action[sl.start + J_FEM_TW] = a * 0.5 * np.sin(phase)  # twist

    # Middle and hind legs: stable wide stance
    for leg_name in ["T2_left", "T2_right", "T3_left", "T3_right"]:
        sl = LEG_SLICES[leg_name]
        action[sl.start + J_FEM] = 0.6
        action[sl.start + J_TIB] = 0.4
        action[sl.start + J_COXA] = 0.3
        action[sl.start + J_COXA_ABD] = -0.2  # wide stable stance

    # Head bobs slightly
    action[8] = -a * 0.2 * np.sin(phase * 0.5)  # head nod

    return action


def feeding_action(t_s, amplitude=1.0, freq=2.0):
    """Proboscis extension: head down, body rocks."""
    action = np.zeros(N_ACTIONS)
    action[:6] = 0.6
    phase = 2.0 * np.pi * freq * t_s
    a = amplitude
    action[8] = -a * 0.4       # head flexes down
    action[10] = a * 0.4 * np.sin(phase)  # abdomen sway
    for leg_name in ["T1_left", "T1_right"]:
        sl = LEG_SLICES[leg_name]
        action[sl.start + J_COXA] = a * 0.6  # front legs forward
        action[sl.start + J_FEM] = a * 0.5
    return action


print("Behavioral primitives defined (high-amplitude)")

## 8. Map Neural Activity → Flybody Actions

For each flybody control step (2 ms), blend behavioral primitives weighted by the
corresponding descending neuron activation levels.

In [ ]:
# Precompute action sequences for TWO flies with distinct behaviors

action_seq_fly1 = np.zeros((N_CONTROL_STEPS, N_ACTIONS))
action_seq_fly2 = np.zeros((N_CONTROL_STEPS, N_ACTIONS))

for step in range(N_CONTROL_STEPS):
    t_s = step * CONTROL_DT_MS / 1000.0

    # ── Fly 1: forward walking + turning (locomotion) ──
    act1 = np.zeros(N_ACTIONS)
    a = fly1_act["forward"][step]
    if a > 0.01:
        act1 += a * tripod_walk_action(t_s, amplitude=1.0, direction=1.0)
    a = fly1_act["turn_left"][step]
    if a > 0.01:
        act1 += a * turning_action(t_s, amplitude=1.0, turn_sign=1.0)
    a = fly1_act["turn_right"][step]
    if a > 0.01:
        act1 += a * turning_action(t_s, amplitude=1.0, turn_sign=-1.0)
    action_seq_fly1[step] = act1

    # ── Fly 2: escape → grooming → backward (defensive) ──
    act2 = np.zeros(N_ACTIONS)
    a = fly2_act["escape"][step]
    if a > 0.01:
        act2 += a * escape_action(t_s, amplitude=1.0)
    a = fly2_act["grooming"][step]
    if a > 0.01:
        act2 += a * grooming_action(t_s, amplitude=1.0)
    a = fly2_act["backward"][step]
    if a > 0.01:
        act2 += a * tripod_walk_action(t_s, amplitude=1.0, direction=-1.0)
    a = fly2_act["feeding"][step]
    if a > 0.01:
        act2 += a * feeding_action(t_s, amplitude=1.0)
    action_seq_fly2[step] = act2

# Clip to valid action range
from flybody.fly_envs import template_task as _tt
_env_tmp = _tt(time_limit=0.1, disable_wings=True)
act_min = _env_tmp.action_spec().minimum
act_max = _env_tmp.action_spec().maximum
del _env_tmp

action_seq_fly1 = np.clip(action_seq_fly1, act_min, act_max)
action_seq_fly2 = np.clip(action_seq_fly2, act_min, act_max)

print(f"Fly 1 actions: {action_seq_fly1.shape}  (walk/turn)")
print(f"Fly 2 actions: {action_seq_fly2.shape}  (escape/groom/backward)")
print(f"Fly 1 active fraction: {100*(np.abs(action_seq_fly1)>0.01).any(axis=1).mean():.0f}%")
print(f"Fly 2 active fraction: {100*(np.abs(action_seq_fly2)>0.01).any(axis=1).mean():.0f}%")

## 9. Build Two-Fly Scene and Record Video

Two flies are placed in the same MuJoCo arena, each driven by different subsets of the
descending neuron activity from the same brain simulation:
- **Fly 1** (left): forward walking, turning, grooming (P9, DNa, aDN1 activations)
- **Fly 2** (right): escape response, backward walking (Giant Fiber, MDN activations)

In [ ]:
from dm_control import mjcf
from dm_control.locomotion.arenas import floors
from flybody.fruitfly.fruitfly import FruitFly
from flybody.tasks.constants import _WALK_PHYSICS_TIMESTEP, _WALK_CONTROL_TIMESTEP

# ── Build the action→ctrl permutation (adhesion-first action → MuJoCo ctrl order) ──
# Action order:  adhesion(6), head(3), abdomen(2), legs(48)
# MuJoCo order:  head(3), abdomen(2), legs(48), adhesion(6)
ACTION_TO_CTRL_LOCAL = np.concatenate([
    np.arange(53, 59),  # action[0:6]  adhesion → ctrl[53:59]
    np.arange(0, 3),    # action[6:9]  head     → ctrl[0:3]
    np.arange(3, 5),    # action[9:11] abdomen  → ctrl[3:5]
    np.arange(5, 53),   # action[11:59] legs    → ctrl[5:53]
])

# ── Create arena with two flies ──
arena = floors.Floor()
wk = dict(use_legs=True, use_wings=False, use_mouth=False, use_antennae=False,
          joint_filter=0.01, adhesion_filter=0.007,
          physics_timestep=_WALK_PHYSICS_TIMESTEP,
          control_timestep=_WALK_CONTROL_TIMESTEP)

fly1 = FruitFly(name='fly1', **wk)
fly2 = FruitFly(name='fly2', **wk)

# Attach fly1 at origin
pos1 = fly1.upright_pose.xpos.copy()
s1 = arena.mjcf_model.worldbody.add('site', pos=pos1)
fly1.create_root_joints(arena.attach(fly1, s1)); s1.remove()

# Attach fly2 offset to the right
FLY_OFFSET = np.array([0.6, 0.0, 0.0])
pos2 = fly2.upright_pose.xpos + FLY_OFFSET
s2 = arena.mjcf_model.worldbody.add('site', pos=pos2)
fly2.create_root_joints(arena.attach(fly2, s2)); s2.remove()

# Floor contact parameters
for geom in arena.ground_geoms:
    geom.friction = (0.5,)
    geom.solref = (0.001, 1)
    geom.solimp = (0.95, 0.99, 0.01)

# Visual settings
arena.mjcf_model.visual.map.znear = 0.001
arena.mjcf_model.visual.map.zfar = 50.0
arena.mjcf_model.statistic.extent = 4.01

# Close-up camera showing both flies
center = (pos1 + pos2) / 2
arena.mjcf_model.worldbody.add(
    'camera', name='duo_view',
    pos=[center[0] + 0.5, center[1] + 0.7, 0.35],
    xyaxes=[-0.8, 0.6, 0, -0.25, -0.33, 0.91],
    fovy=50,
)

# Compile physics
physics = mjcf.Physics.from_mjcf_model(arena.mjcf_model)

# Map each fly's actuators to global ctrl indices
fly1_ctrl = sorted(i for i in range(physics.model.nu) if 'fly1/' in physics.model.actuator(i).name)
fly2_ctrl = sorted(i for i in range(physics.model.nu) if 'fly2/' in physics.model.actuator(i).name)
fly1_global = np.array(fly1_ctrl)[ACTION_TO_CTRL_LOCAL]
fly2_global = np.array(fly2_ctrl)[ACTION_TO_CTRL_LOCAL]

# Find camera
duo_cam = next(i for i in range(physics.model.ncam) if 'duo' in physics.model.cam(i).name)

print(f"Two-fly scene compiled: {physics.model.nu} actuators ({len(fly1_ctrl)} + {len(fly2_ctrl)})")
print(f"Camera: duo_view (id={duo_cam})")

In [ ]:
# ── Run two-fly simulation and record video ──
RENDER_EVERY = 5
RENDER_HEIGHT = 480
RENDER_WIDTH = 640
PHYSICS_SUBSTEPS = 10  # physics_dt=2e-4, control_dt=2e-3 → 10 substeps

frames = []
positions_fly1 = []
positions_fly2 = []

print(f"Running two-fly simulation ({N_CONTROL_STEPS} control steps)...")
t0 = perf_counter()

for step in range(N_CONTROL_STEPS):
    # Apply actions for both flies
    physics.data.ctrl[fly1_global] = action_seq_fly1[step]
    physics.data.ctrl[fly2_global] = action_seq_fly2[step]

    # Step physics
    for _ in range(PHYSICS_SUBSTEPS):
        physics.step()

    # Track positions
    try:
        p1 = physics.named.data.xpos['fly1/thorax'].copy()
        p2 = physics.named.data.xpos['fly2/thorax'].copy()
    except Exception:
        p1 = p2 = np.zeros(3)
    positions_fly1.append(p1)
    positions_fly2.append(p2)

    # Render
    if step % RENDER_EVERY == 0:
        frames.append(physics.render(height=RENDER_HEIGHT, width=RENDER_WIDTH,
                                     camera_id=duo_cam))

    if (step + 1) % 500 == 0:
        print(f"  Step {step+1}/{N_CONTROL_STEPS}")

elapsed = perf_counter() - t0
print(f"\nDone in {elapsed:.2f}s — {len(frames)} frames rendered")

In [ ]:
# Save video as MP4
import imageio.v2 as imageio

video_path = OUTPUT_DIR / "flybody_two_flies.mp4"
effective_fps = 1000.0 / (CONTROL_DT_MS * RENDER_EVERY)
video_fps = min(effective_fps, 60.0)

imageio.mimwrite(str(video_path), frames, fps=video_fps)
print(f"Video saved: {video_path}")
print(f"  {len(frames)} frames at {video_fps:.0f} fps")
print(f"  Simulated time: {TOTAL_MS/1000:.1f}s  |  Playback: {len(frames)/video_fps:.1f}s")

In [ ]:
# Display video inline (Jupyter)
from flybody.utils import display_video
display_video(frames, framerate=int(video_fps))

## 10. Trajectory Analysis

In [ ]:
positions_fly1 = np.array(positions_fly1)
positions_fly2 = np.array(positions_fly2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Top-down trajectory (x-y)
ax = axes[0]
t_ctrl = np.arange(len(positions_fly1)) * CONTROL_DT_MS
stim_idx = int(SPONTANEOUS_MS / CONTROL_DT_MS)

for pos, label, color in [(positions_fly1, 'Fly 1 (walk)', 'royalblue'),
                           (positions_fly2, 'Fly 2 (escape)', 'orangered')]:
    ax.plot(pos[:stim_idx, 0], pos[:stim_idx, 1],
            color=color, alpha=0.4, linewidth=1, linestyle='--')
    if stim_idx < len(pos):
        ax.plot(pos[stim_idx:, 0], pos[stim_idx:, 1],
                color=color, alpha=0.8, linewidth=1.5, label=f'{label} (stim)')
    ax.plot(pos[0, 0], pos[0, 1], 'o', color=color, markersize=6)
    ax.plot(pos[-1, 0], pos[-1, 1], '^', color=color, markersize=6)

ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_title('Top-Down Trajectories')
ax.legend(fontsize=8)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

# Height over time
ax2 = axes[1]
for pos, label, color in [(positions_fly1, 'Fly 1', 'royalblue'),
                           (positions_fly2, 'Fly 2', 'orangered')]:
    ax2.plot(t_ctrl[:len(pos)], pos[:, 2], color=color, linewidth=0.5, label=label)
ax2.axvline(SPONTANEOUS_MS, color='gray', linestyle='--', alpha=0.5, label='Stim onset')
ax2.set_xlabel('Time (ms)')
ax2.set_ylabel('Height Z (m)')
ax2.set_title('Body Height Over Time')
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "trajectory_two_flies.png", dpi=150, bbox_inches="tight")
plt.show()

for name, pos in [("Fly 1", positions_fly1), ("Fly 2", positions_fly2)]:
    d = np.linalg.norm(pos[-1, :2] - pos[0, :2])
    print(f"{name} displacement: {d*1000:.2f} mm")

## 11. Save Spike Data for Reuse

In [ ]:
# Save spike data
spikes_path = OUTPUT_DIR / "spikes_integration.h5"
sort_idx = np.argsort(all_spike_times)

with h5py.File(spikes_path, "w") as f:
    pop = f.create_group("spikes/internal")
    ds = pop.create_dataset("timestamps", data=all_spike_times[sort_idx])
    ds.attrs["units"] = "ms"
    pop.create_dataset("node_ids", data=all_spike_ids[sort_idx].astype(np.uint64))
    pop.attrs["sorting"] = "by_time"

    # Save motor neuron spike trains separately
    motor_grp = f.create_group("motor_neurons")
    for name, steps in motor_spike_trains.items():
        times_ms = np.array(steps, dtype=np.float64) * dt
        motor_grp.create_dataset(name, data=times_ms)
        motor_grp[name].attrs["flywire_id"] = MOTOR_NEURONS[name]
        motor_grp[name].attrs["node_id"] = motor_node_ids[name]

    # Save experiment config
    meta = f.create_group("metadata")
    meta.attrs["spontaneous_ms"] = SPONTANEOUS_MS
    meta.attrs["stimulated_ms"] = STIMULATED_MS
    meta.attrs["total_ms"] = TOTAL_MS
    meta.attrs["experiments"] = str(ACTIVE_EXPERIMENTS)

# Save action sequences
np.save(OUTPUT_DIR / "action_seq_fly1.npy", action_seq_fly1)
np.save(OUTPUT_DIR / "action_seq_fly2.npy", action_seq_fly2)

print(f"Saved: {spikes_path}")
print(f"Saved: fly1/fly2 action sequences")
print(f"\nAll outputs in: {OUTPUT_DIR.resolve()}")